# Browser media operations
Run this notebook cell by cell in a disposable local JupyterLab project, with the notebook at the Jupyter server root. The kernel's working directory must be that root. Each browser call returns promptly; wait for its receipt to complete, then inspect it in the next cell. The example makes a tiny PNG locally and never asks for device permission.

## Make an exact local media reference

In [ ]:
from pathlib import Path
from uuid import uuid4
import hashlib
from PIL import Image

server_root = Path.cwd().resolve()  # Start JupyterLab with this disposable project as its server root.
source = server_root / f'browser-media-source-{uuid4().hex[:8]}.png'
Image.new('RGB', (2, 2), 'red').save(source)
source_ref = {'path': source.name, 'sha256': hashlib.sha256(source.read_bytes()).hexdigest()}
source_ref

## browser_capabilities

In [ ]:
from nbinlineai.tools import browser_capabilities
capabilities = browser_capabilities()
capabilities

In [ ]:
capabilities.status, capabilities.result, capabilities.error

## save_media

In [ ]:
from nbinlineai.tools import save_media
saved = save_media(source_ref, save_to='auto')
saved

In [ ]:
saved.status, saved.operation_id, saved.media, saved.error

## operation_status

In [ ]:
from nbinlineai.tools import operation_status
assert saved.status == 'completed', 'Wait for the save receipt, then rerun this cell.'
lookup = operation_status(saved.operation_id)
lookup

In [ ]:
lookup.status, lookup.result, lookup.error  # The lookup completes once; result holds the target snapshot.

## cancel_operation

In [ ]:
from nbinlineai.tools import cancel_operation
cancelled = cancel_operation(saved.operation_id)
cancelled

In [ ]:
cancelled.status, cancelled.media, cancelled.error  # Cancelling a completed operation does not delete its file.

## release_media

In [ ]:
picture = Image.open(server_root / saved.media['path'])
picture.load()
from nbinlineai.tools import release_media
released = release_media(saved.media['media_id'])
released

In [ ]:
released.status, (server_root / saved.media['path']).exists(), picture.size, released.error

In [ ]:
saved_path = server_root / saved.media['path']
assert saved_path.exists() and picture.size == (2, 2)
saved_path.unlink()  # Remove only the exact file this example saved.
source.unlink(missing_ok=True)
not saved_path.exists() and not source.exists()